# Walkthrough of the LPIPS Perceptual Loss and PatchGAN Adversarial Loss Functions

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vgg16, VGG16_Weights # build lpips from a pretrained VGG
import warnings
warnings.filterwarnings("ignore")

In [5]:
class LPIPS(nn.Module):
    def __init__(self,
                 pretrained_backbone=True,
                 train_backbone=False,
                 use_dropout=True,
                 img_range="minus_one_to_one",
                 pretrained_weights=None):

        self.pretrained_backbone = pretrained_backbone
        self.train_backbone = train_backbone
        self.use_dropout = use_dropout
        self.img_range = img_range

        # load a pretrained VGG backbone and its channel pattern
        vgg_model = vgg16(weights=VGG16_Weights.IMAGENET1K_V1 if pretrained_backbone else None)

        self.channels = [64, 128, 256, 512, 512]
        self.layer_groups = [(0, 3), (4, 8), (9, 15), (16, 22), (23, 29)] # chunk the model up, and grab features from each layer chunk

        if not train_backbone:
            for param in vgg_model.parameters():
                param.requires_grad_(False)

        self.scale_constants(img_range)

    def scale_constants(self, range="minus_one_to_one"):
        if range not in ["zero_to_one", "minus_one_to_one"]:
            raise ValueError("Indicate if images are zero_to_one [0, 1] or minus_one_to_one [-1, 1]")

        # standard imagenet mean and std assume images are in [0,1]
        imagenet_mean = torch.tensor([0.485, 0.456, 0.406])
        imagenet_std = torch.tensor([0.229, 0.224, 0.225])

        if range == "minus_one_to_one":
            # [0-1] * 2 => [0,2] - 1 => [-1,1]
            imagenet_std = imagenet_std * 2
            imagenet_mean = imagenet_mean * 2 - 1

        imagenet_mean = imagenet_mean.reshape(1, 3, 1, 1)
        imagenet_std = imagenet_std.reshape(1, 3, 1, 1)

        self.register_buffer("mean", imagenet_mean)
        self.register_buffer("std", imagenet_std)

In [6]:
model = LPIPS()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1